In [ ]:
import json, numpy as np, pandas as pd
from datetime import datetime, timezone

PATH = "arData/stability.json"

with open(PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

def extract_window_metrics(rec):
    metrics = rec.get("metrics", {})
    interactions = metrics.get("interactions", rec.get("interactions", [])) or []
    dev_orients = metrics.get("deviceOrientations", [])
    # letzten Klick auf start-countdown-btn nehmen
    start_clicks = [int(ev["timestamp"]) for ev in interactions
                    if isinstance(ev, dict)
                    and (ev.get("type") or "").lower() == "click"
                    and ev.get("elementId") == "start-countdown-btn"
                    and isinstance(ev.get("timestamp"), (int, float))]
    if not start_clicks:
        return None
    start = max(start_clicks)
    end = start + 20000  # 20 s Fenster

    betas = [float(d["beta"]) for d in dev_orients
             if isinstance(d, dict)
             and isinstance(d.get("timestamp"), (int, float))
             and isinstance(d.get("beta"), (int, float))
             and start <= int(d["timestamp"]) <= end]
    if len(betas) < 5:
        return None

    w = np.array(betas, dtype=float)
    abs_w = np.abs(w)
    dev90 = np.abs(w - 90.0)

    # Interquartilsabstand (IQR) als Streuungsmaß
    q25_abs, q75_abs = np.percentile(abs_w, [25, 75])
    q25_dev, q75_dev = np.percentile(dev90, [25, 75])

    return {
        "mean_abs_beta": float(np.mean(abs_w)),
        "median_abs_beta": float(np.median(abs_w)),
        "mean_dev_from_90": float(np.mean(dev90)),
        "median_dev_from_90": float(np.median(dev90)),
        "spread_abs_beta": float(q75_abs - q25_abs),         # IQR(|β|)
        "spread_dev_from_90": float(q75_dev - q25_dev),      # IQR(|β-90|)
        "n": int(len(w))
    }

rows = []
for i, rec in enumerate(data):
    m = extract_window_metrics(rec)
    if m:
        rows.append({"X": i, **m})
df = pd.DataFrame(rows)
df.head()


In [ ]:

out_csv = "haltewinkel_20s_metrics_pro_teilnehmer.csv"
df.to_csv(out_csv, index=False)
out_csv

In [ ]:
summary = {
    "Durchschnitt – Mediane |β| [°]": df["median_abs_beta"].mean(),
    "Durchschnitt – Mittelwerte |β| [°]": df["mean_abs_beta"].mean(),
    "Durchschnitt – Abweichung von 90° (Mediane |β-90|) [°]": df["median_dev_from_90"].median(),
    "Durchschnitt – Streuung |β| (IQR) [°]": df["spread_abs_beta"].mean(),
    "Durchschnitt – Streuung Abweichung von 90° (IQR) [°]": df["spread_dev_from_90"].mean(),
    "Teilnehmer (n)": len(df)
}
pd.DataFrame([{k: round(v, 2) if isinstance(v, (int, float)) else v for k, v in summary.items()}])


In [ ]:

import matplotlib.pyplot as plt

plt.figure()
plt.hist(df["median_abs_beta"].dropna(), bins=15)
plt.xlabel("Ergonomischer Haltewinkel je Teilnehmer – Median |β| [°]")
plt.ylabel("Häufigkeit")
plt.title("Verteilung der ergonomischen Haltewinkel (Median |β|)")
plt.show()

In [ ]:

import matplotlib.pyplot as plt

plt.figure()
plt.boxplot(df["median_abs_beta"].dropna(), vert=True)
plt.ylabel("Median |β| [°]")
plt.title("Boxplot der ergonomischen Haltewinkel (Median |β|)")
plt.show()